[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/28_gqa.ipynb)

# 🟡 Medium: Grouped-Query Attention

*Attention & Transformers*
Implement **Grouped-Query Attention** (GQA) as an `nnx.Module`: multi-head
attention where the key/value heads are *fewer* than the query heads and each
KV head is shared by a contiguous group of query heads.

With $H$ query heads, $H_{kv}$ key/value heads and $r = H / H_{kv}$:

$$\text{head}_h = \mathrm{softmax}\!\left(\frac{Q_h K_{\lfloor h/r \rfloor}^\top}{\sqrt{d_h}}\right) V_{\lfloor h/r \rfloor}$$

$$\text{out} = \mathrm{concat}(\text{head}_0, \dots, \text{head}_{H-1})\, W_o$$

$H_{kv} = H$ is ordinary MHA; $H_{kv} = 1$ is Multi-Query Attention.

### Rules
- Subclass `nnx.Module`. Do **not** use `nnx.MultiHeadAttention`,
  `nnx.dot_product_attention` or `jax.nn.dot_product_attention`
- Signature: `GroupedQueryAttention(d_model, num_heads, num_kv_heads, *, rngs)`
- Parameters — all `nnx.Param`, all `(din, dout)`, **no biases**:
  - `self.w_q`, `self.w_o`: `(d_model, d_model)`
  - `self.w_k`, `self.w_v`: `(d_model, num_kv_heads * head_dim)`
- `head_dim = d_model // num_heads`; `num_heads % num_kv_heads == 0`
- `__call__(x)` maps `(B, T, d_model) -> (B, T, d_model)`, bidirectional
  self-attention (no causal mask)
- Split heads as `(B, T, H, head_dim) -> transpose -> (B, H, T, head_dim)`,
  i.e. head `h` owns channels `[h*head_dim : (h+1)*head_dim]`
- Scale scores by `1 / sqrt(head_dim)` — **not** `1 / sqrt(d_model)`
- Query head `h` reads KV head `h // r`, so the KV heads are repeated
  **interleaved** `(0,0,1,1)`, not tiled `(0,1,0,1)`

### Why GQA exists: the KV cache, not the FLOPs
GQA saves almost no compute. After the repeat, the score and value matmuls are
byte-for-byte the same size as MHA; only the two `d_model x d_model` KV
projections shrink. The win is entirely **decode-time memory**.

At generation time every past token's K and V must be kept. Per token per layer
the cache costs $2 \cdot H_{kv} \cdot d_h \cdot \text{bytes}$. For a 70B-class
model (80 layers, $H = 64$, $d_h = 128$, fp16):

| | per token | 4k context | 4k ctx, batch 8 |
|---|---|---|---|
| MHA ($H_{kv}=64$) | 2.6 MB | 10.7 GB | 86 GB |
| GQA ($H_{kv}=8$) | 328 KB | 1.3 GB | 10.7 GB |
| MQA ($H_{kv}=1$) | 41 KB | 168 MB | 1.3 GB |

Autoregressive decoding is memory-**bandwidth** bound: each new token reads the
entire cache from HBM to produce one token. Shrinking the cache 8x shrinks the
per-step read 8x, so it is close to an 8x decode speedup as well as the reason a
long context fits in memory at all. MQA takes this to the limit but measurably
degrades quality; GQA is the interpolation that keeps ~MHA quality — and it is
cheap to *uptrain* an existing MHA checkpoint into GQA by mean-pooling the KV
heads within each group.

### The trap
`jnp.repeat(k, r, axis=1)` and `jnp.tile(k, (1, r, 1, 1))` produce arrays of
identical shape and different content. Pick the wrong one and every test that
only checks shapes passes, the model trains, and it silently pairs query head 1
with the KV head meant for query head `r`. When you later load real GQA weights,
the output is garbage with a perfectly valid shape.

A production kernel skips the `repeat` entirely: reshape Q to
`(B, H_kv, r, T, d_h)` and `einsum` against the un-repeated K, so the shared KV
is read once instead of materialised `r` times. Same math, less memory traffic.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class GroupedQueryAttention(nnx.Module):
    """Multi-head attention with num_kv_heads < num_heads shared KV heads."""

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        num_kv_heads: int,
        *,
        rngs: nnx.Rngs,
    ):
        pass  # Replace this

    def __call__(self, x):
        """(B, T, d_model) -> (B, T, d_model)"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

D, H = 512, 8
x = jax.random.normal(jax.random.key(0), (1, 16, D))

for kv in (8, 4, 1):
    m = GroupedQueryAttention(D, H, kv, rngs=nnx.Rngs(params=0))
    n_params = sum(p.size for p in jax.tree.leaves(nnx.state(m, nnx.Param)))
    cache_per_token = 2 * kv * (D // H) * 2          # K and V, fp16 bytes
    print(
        f"kv_heads={kv:>2}  w_k={str(m.w_k.shape):>12}  "
        f"params={n_params:>8,}  KV cache/token/layer={cache_per_token:>6} B  "
        f"out={m(x).shape}"
    )
print("\n-> output shape never changes; the cache shrinks linearly in kv_heads")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("gqa")

# hint("gqa")      # stuck? nudge without the answer
# solution("gqa")  # spoiler: the reference implementation